# Reproduce paper tables

Regenerates the manuscript **main tables (Table 1–5)** and **supplementary tables
(Table S1–S3, S6–S7)** from the derived summary files in `../data/`.
No GPU, checkpoints, or network are required.

Each table is assembled as a `pandas.DataFrame`, written to `../tables/` as a CSV, and
all tables are also written to a single multi-sheet workbook `../tables/paper_tables.xlsx`.
Sanity-check `assert`s verify that the numbers match the manuscript narrative
(e.g. the 3/6 ensemble wins on Polaris, the k=5 composition).

- **Table 1** — 22 MTL-training endpoints, grouped by ADMET mechanism
- **Table 2** — headline ensemble result (21-task norm_score & mean rank, within/cross)
- **Table 3** — the five final k=5 members: solo score, Leave-One-Out, Shapley
- **Table 4** — per-task held-out-test of the final ensemble (21 tasks)
- **Table 5** — external validation: LateFusion-k5 rank on Polaris ADME-Fang
- **Table S1** — per-backbone per-task (8 base models)
- **Table S2** — representation-level aggregation (MAP/mean/max)
- **Table S3** — ensemble-size sweep (top-20 mean, k=1..10)
- **Table S6** — Polaris ADME-Fang dataset specification
- **Table S7** — external per-model per-endpoint Pearson r


In [1]:
import json, os
import numpy as np, pandas as pd

DATA="../data"; OUT="../tables"; os.makedirs(OUT, exist_ok=True)
def lj(fn):
    with open(os.path.join(DATA,fn)) as f: return json.load(f)
def lc(fn):
    return pd.read_csv(os.path.join(DATA,fn))

td   = lj("tables_data.json")
gt   = lj("ground_truth.json")
pt   = lj("pertask.json")
stat = lj("stat_tests.json")

TABLES={}   # name -> DataFrame, collected for the xlsx workbook
def register(name, df, index=False):
    TABLES[name]=(df,index)
    df.to_csv(os.path.join(OUT, name+".csv"), index=index)
    return df
print("loaded:", sorted(os.listdir(DATA)))


loaded: ['DATA_MANIFEST.md', 'additions.json', 'all1023.csv', 'ground_truth.json', 'pertask.json', 'polaris_all_model_test_preds.csv', 'polaris_ensemble_pertask_val_pearson.csv', 'polaris_leaderboard_full.json', 'polaris_official_overall.csv', 'polaris_official_pertask_pearson.csv', 'polaris_official_submitted_rank.csv', 'reconciliation_flags.json', 'solo10_within_cross.json', 'stat_tests.json', 'tables_data.json', 'uniform_val_test_unified.csv']


## Table 1 — MTL-training endpoints grouped by ADMET mechanism

In [2]:
t1 = pd.DataFrame(td["T1"])[["group","tasks","rationale"]]
t1.columns = ["ADMET group","Endpoints","Grouping rationale"]
register("table1_endpoint_groups", t1)
t1

## Table 2 — Headline ensemble result (21 valid tasks)

In [3]:
t2 = pd.DataFrame(td["T2"], columns=["Track","Mean norm_score","Mean rank","#1","top-3","top-5"])
register("table2_headline", t2)
t2

## Table 3 — Final k=5 members: solo, Leave-One-Out, Shapley

Positive LOO Δ = removing the member **hurts** the ensemble. GraphMVP·STL has a slightly
negative LOO (removal marginally helps this one fixed coalition) yet a positive Shapley
(it contributes across coalitions on average).

In [4]:
t3 = pd.DataFrame(td["T3"], columns=[
    "Member","Solo (within)","Solo (cross)","LOO Δ (within)","LOO Δ (cross)",
    "Shapley (within)","Shapley (cross)"])
register("table3_members", t3)
t3

## Table 4 — Per-task held-out-test of the final k=5 ensemble (21 tasks)

In [5]:
t4 = pd.DataFrame(td["T4"], columns=[
    "Task","Metric","Within score","Within norm","Within rank",
    "Cross score","Cross norm","Cross rank"])
register("table4_pertask_ensemble", t4)
t4

## Table 5 — External validation: LateFusion-k5 on Polaris ADME-Fang

Official Polaris Hub leaderboard rank of our submitted seed-averaged ensemble.

In [6]:
t5 = lc("polaris_official_submitted_rank.csv")
t5 = t5.rename(columns={"benchmark":"Endpoint","slug":"Polaris slug","our_pearson":"LateFusion-k5 (Pearson r)",
                        "rank":"Rank","n_entries":"# submissions","top_model":"Leaderboard best model",
                        "top_pearson":"Best Pearson r","gap_to_top":"Gap to best"})
register("table5_polaris_rank", t5)
# sanity: 2 firsts, mean pearson
n_first = int((t5["Rank"]==1).sum())
assert n_first==2, n_first
print(f"first-place endpoints: {n_first} (RLM_CLint, HPPB)")
print(f"mean Pearson r: {t5['LateFusion-k5 (Pearson r)'].mean():.4f}")
t5

first-place endpoints: 2 (RLM_CLint, HPPB)
mean Pearson r: 0.8186


## Table S1 — Per-backbone, per-task norm_score (8 base models)

In [7]:
cols_s1 = ["Group","Task","n","Metric"] + [
    {"mtl_dmpnn":"D-MPNN·MTL","mtl_chemeleon":"CheMeleon·MTL","mtl_graphmvp":"GraphMVP·MTL",
     "mtl_molformer":"MoLFormer·MTL","stl_dmpnn":"D-MPNN·STL","stl_chemeleon":"CheMeleon·STL",
     "stl_graphmvp":"GraphMVP·STL","stl_molformer":"MoLFormer·STL"}[c] for c in td["cols_s1"]]
s1 = pd.DataFrame(td["S1"], columns=cols_s1)
register("tableS1_per_backbone_pertask", s1)
s1.head(6)

## Table S2 — Representation-level aggregation (MAP vs mean vs max)

In [8]:
cols_s2 = ["Group","Task","Metric"] + [
    {"mtl_map":"MAP·MTL","mtl_mean":"mean·MTL","mtl_max":"max·MTL",
     "stl_map":"MAP·STL","stl_mean":"mean·STL","stl_max":"max·STL"}[c] for c in td["cols_s2"]]
s2 = pd.DataFrame(td["S2"], columns=cols_s2)
register("tableS2_representation_aggregation", s2)
s2.head(6)

## Table S3 — Ensemble-size sweep (top-20 mean norm_score)

The **combined Δ** column (within Δ + cross Δ) is the basis for the k=5 elbow shown in
Figure 3b: it is the last value above ε = 0.01 at k=4→5 (+0.0104) and drops below ε at
k=5→6 (+0.0029).

In [9]:
s3 = pd.DataFrame(td["S3"], columns=[
    "k","Within top-20 mean","Within \u0394","Cross top-20 mean","Cross \u0394"])
# combined marginal Δ = within Δ + cross Δ (matches Figure 3b; per-track values are half of this)
def _num(x):
    try: return float(x)
    except (TypeError, ValueError): return np.nan
s3["Combined \u0394 (within+cross)"] = [
    round(_num(w)+_num(c), 4) if not (np.isnan(_num(w)) or np.isnan(_num(c))) else np.nan
    for w,c in zip(s3["Within \u0394"], s3["Cross \u0394"])]
register("tableS3_ksweep", s3)
# sanity: combined marginal at k=5 = 0.0104 (>eps), at k=6 = 0.0029 (<eps)
assert abs(s3.loc[s3.k==5,"Combined \u0394 (within+cross)"].iloc[0]-0.0104)<1e-6
assert abs(s3.loc[s3.k==6,"Combined \u0394 (within+cross)"].iloc[0]-0.0029)<1e-6
print("combined Δ k4->5:", s3.loc[s3.k==5,"Combined \u0394 (within+cross)"].iloc[0],
      "| k5->6:", s3.loc[s3.k==6,"Combined \u0394 (within+cross)"].iloc[0])
s3

combined Δ k4->5: 0.0104 | k5->6: 0.0029


## Table S6 — Polaris ADME-Fang dataset specification

In [10]:
ds=[("HLM_CLint","adme-fang-hclint-reg-v1","LOG_HLM_CLint",2230,576),
    ("RLM_CLint","adme-fang-rclint-reg-v1","LOG_RLM_CLint",2220,559),
    ("MDR1_ER","adme-fang-perm-reg-v1","LOG_MDR1-MDCK_ER",1920,483),
    ("Solubility","adme-fang-solu-reg-v1","LOG_SOLUBILITY",1578,400),
    ("HPPB","adme-fang-hppb-reg-v1","LOG_HPPB",126,34),
    ("RPPB","adme-fang-rppb-reg-v1","LOG_RPPB",111,24)]
s6=pd.DataFrame(ds, columns=["Endpoint","Polaris slug","Target variable","n (train+val)","n (test)"])
register("tableS6_polaris_datasets", s6)
s6

## Table S7 — External per-model, per-endpoint Pearson r

The basis for the main-text 3/6-win claim. Best value in each row is flagged.

In [11]:
pm = lc("polaris_official_pertask_pearson.csv").set_index("task")
models=["mtl_dmpnn","mtl_molformer","stl_chemeleon","stl_graphmvp","stl_map","ensemble"]
mlab={"mtl_dmpnn":"D-MPNN·MTL","mtl_molformer":"MoLFormer·MTL","stl_chemeleon":"CheMeleon·STL",
      "stl_graphmvp":"GraphMVP·STL","stl_map":"MAP·STL","ensemble":"LateFusion-k5"}
order=["Fang_HLM_CLint","Fang_RLM_CLint","Fang_MDR1_ER","Fang_Solubility","Fang_HPPB","Fang_RPPB"]
s7=pm.loc[order, models].round(4).copy()
s7.insert(0,"best_model",[mlab[s7.loc[t].idxmax()] for t in order])
s7.index=[t.replace("Fang_","") for t in order]
s7.columns=["Best model"]+[mlab[m] for m in models]
register("tableS7_polaris_per_model", s7.reset_index().rename(columns={"index":"Endpoint"}))

# sanity check: ensemble wins exactly 3/6, RPPB best = MoLFormer·MTL (not D-MPNN)
ens_wins = sum(1 for t in order if pm.loc[t, models].idxmax()=="ensemble")
rppb_best = mlab[pm.loc["Fang_RPPB", models].idxmax()]
assert ens_wins==3, ens_wins
assert rppb_best=="MoLFormer·MTL", rppb_best
print(f"ensemble wins {ens_wins}/6 endpoints; RPPB best single = {rppb_best}")
s7

ensemble wins 3/6 endpoints; RPPB best single = MoLFormer·MTL


## Write all tables to a multi-sheet workbook

In [12]:
with pd.ExcelWriter(os.path.join(OUT,"paper_tables.xlsx"), engine="openpyxl") as xw:
    sheetmap={"table1_endpoint_groups":"T1_groups","table2_headline":"T2_headline",
        "table3_members":"T3_members","table4_pertask_ensemble":"T4_pertask",
        "table5_polaris_rank":"T5_polaris_rank","tableS1_per_backbone_pertask":"S1_per_backbone",
        "tableS2_representation_aggregation":"S2_aggregation","tableS3_ksweep":"S3_ksweep",
        "tableS6_polaris_datasets":"S6_datasets","tableS7_polaris_per_model":"S7_per_model"}
    for name,(df,idx) in TABLES.items():
        df.to_excel(xw, sheet_name=sheetmap.get(name,name)[:31], index=idx)
print("wrote", len(TABLES), "tables ->", sorted(os.listdir(OUT)))

wrote 10 tables -> ['paper_tables.xlsx', 'table1_endpoint_groups.csv', 'table2_headline.csv', 'table3_members.csv', 'table4_pertask_ensemble.csv', 'table5_polaris_rank.csv', 'tableS1_per_backbone_pertask.csv', 'tableS2_representation_aggregation.csv', 'tableS3_ksweep.csv', 'tableS6_polaris_datasets.csv', 'tableS7_polaris_per_model.csv']


All tables written to `../tables/` (CSV each + `paper_tables.xlsx`).